# Furniture-Aware Training (가구 인식 통합 학습)

**목표:** 빌트인 가구 환경에서 정확한 부위 분류

**클래스 (10개):**
- 부위: wall, ceiling, floor, window, door
- 빌트인 가구: cabinet, kitchen_appliance, countertop_sink, kitchen_island, shelf

**전략:**
- yolov11l (47M)
- imgsz 640
- batch 8 (A100)
- 80ep + Stage 2 fine-tune 20ep
- multi-stage + Drive 자동 저장

**Drive 준비:**
1. `furniture_aware.zip` (예상 ~3-5GB)

**예상 시간:** A100 약 8~10시간

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu numpy pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, shutil, zipfile
from pathlib import Path

# ★ 본인 계정 폴더 ★
DRIVE_DIR = Path('/content/drive/MyDrive/drone_inspect')  # A 계정 (또는 'drone_inspect_A' for C)
ZIP_NAME = 'furniture_aware.zip'

WORK = Path('/content/furniture_aware')
WORK.mkdir(parents=True, exist_ok=True)

zip_path = DRIVE_DIR / ZIP_NAME
assert zip_path.exists(), f'{zip_path} 없음'
ds_dir = WORK / 'furniture_aware'
if not (ds_dir / 'images' / 'val').exists():
    if ds_dir.exists():
        shutil.rmtree(ds_dir, ignore_errors=True)
    print('Extracting...')
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            rel = m.replace('\\', '/')
            target = WORK / rel
            if rel.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    print(f'Done: {ds_dir}')
for s in ['train', 'val', 'test']:
    p = ds_dir / 'images' / s
    if p.exists():
        print(f'  {s}: {len(list(p.glob("*")))} images')

In [ ]:
yaml_text = f'''path: {ds_dir}
train: images/train
val: images/val
test: images/test

nc: 10
names:
  0: wall
  1: ceiling
  2: floor
  3: window
  4: door
  5: cabinet_builtin
  6: kitchen_appliance
  7: countertop_sink
  8: kitchen_island
  9: shelf
'''
data_yaml = WORK / 'furniture_aware.yaml'
data_yaml.write_text(yaml_text)
print(data_yaml.read_text())

In [ ]:
AUTOSAVE = DRIVE_DIR / 'furniture_aware_autosave'
AUTOSAVE.mkdir(parents=True, exist_ok=True)
PROJECT = Path('/content/runs/furniture_aware')
PROJECT.mkdir(parents=True, exist_ok=True)

import time, threading
stop_flag = [False]
def autosave():
    while not stop_flag[0]:
        time.sleep(300)
        for s in ['stage1', 'stage2']:
            for src in [PROJECT/s/'weights/last.pt', PROJECT/s/'weights/best.pt']:
                if src.exists():
                    try:
                        shutil.copy2(src, AUTOSAVE / f'{s}_{src.name}')
                    except Exception:
                        pass
        print(f'[autosave] {time.strftime("%H:%M:%S")}')
threading.Thread(target=autosave, daemon=True).start()

In [ ]:
from ultralytics import YOLO

print('=== Stage 1 (yolov11l + 640 + batch=8 + 80ep) ===')
model_s1 = YOLO('yolo11l.pt')
model_s1.train(
    data=str(data_yaml),
    epochs=80,
    batch=8,
    imgsz=640,
    cache='disk',
    workers=4,
    optimizer='AdamW',
    lr0=1e-3,
    lrf=0.01,
    cos_lr=True,
    patience=25,
    warmup_epochs=3,
    close_mosaic=20,
    freeze=0,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.5,
    shear=2.0, perspective=0.001,
    flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.15, copy_paste=0.5,
    erasing=0.0,
    multi_scale=0.15,
    save_period=5,
    plots=True,
    project=str(PROJECT),
    name='stage1',
    exist_ok=True
)
print('Stage 1 done.')

In [ ]:
stage1_best = PROJECT / 'stage1' / 'weights' / 'best.pt'
print('=== Stage 2 (lr=1e-5, freeze=10, 20ep) ===')
model_s2 = YOLO(str(stage1_best))
model_s2.train(
    data=str(data_yaml),
    epochs=20,
    batch=8,
    imgsz=640,
    cache='disk',
    workers=4,
    optimizer='AdamW',
    lr0=1e-5,
    lrf=0.01,
    cos_lr=True,
    patience=12,
    warmup_epochs=1,
    close_mosaic=15,
    freeze=10,
    mosaic=0.5, mixup=0.0, copy_paste=0.2,
    save_period=5,
    plots=True,
    project=str(PROJECT),
    name='stage2',
    exist_ok=True
)
stop_flag[0] = True
print('Stage 2 done.')

In [ ]:
stage1_best = PROJECT / 'stage1' / 'weights' / 'best.pt'
stage2_best = PROJECT / 'stage2' / 'weights' / 'best.pt'
best_path = stage2_best if stage2_best.exists() else stage1_best
best_model = YOLO(str(best_path))
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)
onnx_path = best_path.with_suffix('.onnx')
metrics = best_model.val(data=str(data_yaml), imgsz=640, batch=8)
print(f'\n=== Furniture-Aware 최종 ===')
print(f'  mAP50:    {metrics.box.map50:.4f}')
print(f'  mAP50-95: {metrics.box.map:.4f}')
print(f'  precision: {metrics.box.mp:.4f}')
print(f'  recall:    {metrics.box.mr:.4f}')
print(f'  0.9? {"YES" if metrics.box.map50 >= 0.9 else "NO"}')
for i, n in enumerate(['wall','ceiling','floor','window','door','cabinet','appliance','counter','island','shelf']):
    if i < len(metrics.box.maps):
        print(f'  {n}: mAP50-95={metrics.box.maps[i]:.4f}')
OUT = DRIVE_DIR / 'furniture_aware_results'
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(onnx_path, OUT / 'furniture_aware.onnx')
shutil.copy2(best_path, OUT / 'furniture_aware_best.pt')
for s in ['stage1', 'stage2']:
    rcsv = PROJECT / s / 'results.csv'
    if rcsv.exists():
        shutil.copy2(rcsv, OUT / f'{s}_results.csv')
print('Saved:', OUT)